In [1]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

# Importing parser

from langchain_core.output_parsers import StrOutputParser

In [2]:
load_dotenv()

True

In [3]:
model = GoogleGenerativeAI(
    model = 'gemini-1.5-flash'
)

In [4]:
template = PromptTemplate(
    template = 'Generate 5 interesting facts about {topic}',
    input_variables = ['topic']
)


parser = StrOutputParser()

# 1) Simple Chain

In [5]:
# Creating a chain

chain = template | model | parser

response = chain.invoke(
    {'topic': 'Ancient India'}
)

print(response)

1. **Ancient India boasted sophisticated urban planning:** Cities like Mohenjo-daro and Harappa, part of the Indus Valley Civilization (circa 3300-1300 BCE), featured remarkably advanced urban planning centuries before similar developments in other parts of the world.  These cities included grid-like street layouts, sophisticated drainage systems, and standardized brick sizes, showcasing a high degree of societal organization and engineering prowess.

2. **Zero's invention revolutionized mathematics:**  The concept of zero, a fundamental element of modern mathematics, is widely credited to ancient Indian mathematicians.  Its development allowed for the creation of the decimal system, significantly advancing mathematical calculations and paving the way for future scientific discoveries.

3. **The concept of democracy had early roots in India:**  While not exactly the same as modern democracy, the ancient Indian republic of Vaishali (circa 6th century BCE) employed a system of governance

In [6]:
# Visualize the chain

chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
  +--------------------+   
  | GoogleGenerativeAI |   
  +--------------------+   
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


# Simple Chain 2: Here, we will call the LLM model twice

In [7]:
temp1 = PromptTemplate(
    template = "Explain about {topic} in detail, in simple manner",
    input_variables = ['topic']
)

temp2 = PromptTemplate(
    template = '''Convert below given paragraph into a 5 line summary
    {paragraph}''',
    input_variable = ['paragraph']
)

In [8]:
# Creating chain

chain = temp1 | model | parser | temp2 | model | parser

response = chain.invoke(
    {'topic': 'Ancient India'}
)

In [9]:
print(response)

Ancient India spans millennia, encompassing diverse cultures and empires.  The Indus Valley Civilization (3300-1300 BCE) showcased advanced urban planning and technology.  The Vedic Period (1500-500 BCE) saw the development of Sanskrit and the caste system.  Powerful empires like the Mauryan and Gupta (322 BCE-550 CE) brought periods of unification and cultural flourishing.  Key aspects include Hinduism, Buddhism, and remarkable achievements in art, literature, and science.  Later periods saw the rise of regional powers and the influence of Islam.



In [10]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
  +--------------------+   
  | GoogleGenerativeAI |   
  +--------------------+   
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

In [11]:
chain

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Explain about {topic} in detail, in simple manner')
| GoogleGenerativeAI(model='gemini-1.5-flash', google_api_key=SecretStr('**********'), client=genai.GenerativeModel(
      model_name='models/gemini-1.5-flash',
      generation_config={},
      safety_settings={},
      tools=None,
      system_instruction=None,
      cached_content=None
  ))
| StrOutputParser()
| PromptTemplate(input_variables=['paragraph'], input_types={}, partial_variables={}, template='Convert below given paragraph into a 5 line summary\n    {paragraph}')
| GoogleGenerativeAI(model='gemini-1.5-flash', google_api_key=SecretStr('**********'), client=genai.GenerativeModel(
      model_name='models/gemini-1.5-flash',
      generation_config={},
      safety_settings={},
      tools=None,
      system_instruction=None,
      cached_content=None
  ))
| StrOutputParser()

# Parallel Chain

In [12]:
model_gemini = GoogleGenerativeAI(
    model = 'gemini-2.5-flash-lite'
)

model_gpt = ChatOpenAI(
    model = 'gpt-3.5-turbo'
)

In [13]:
temp1 = PromptTemplate(
    template = '''Generate short and simple notes from the below given paragraph.
    {paragraph}''',
    input_variables = ['paragraph']
)

temp2 = PromptTemplate(
    template = '''
    From the below given paragraph, enerate 5 MCQs with 4 options and its correct answer
    {paragraph}''',
    input_variables = ['paragraph']
)

temp3 = PromptTemplate(
    template = '''
    Merge the below given paragraph and MCQs into a single document
    {notes}
    {mcq}''',
    input_variables = ['notes', 'mcq']
)

parser = StrOutputParser()

In [14]:
from langchain.schema.runnable import RunnableParallel

# We can create multiple parallel models using RunnableParallel.
# We will learn what are Runnables in our next session

In [15]:
# Creating chains

parallel_chain = RunnableParallel(
    {
        'notes': temp1 | model_gemini | parser,
        'mcq': temp2 | model_gpt | parser
    }
)

merge_chain = temp3 | model_gemini | parser

combined_chain = parallel_chain | merge_chain

In [16]:
text = '''
A Decision Tree helps us to make decisions by mapping out different choices and their possible outcomes. 
It’s used in machine learning for tasks like classification and prediction.
In this article, we’ll see more about Decision Trees, their types and other core concepts.

A Decision Tree helps us make decisions by showing different options and how they are related. 
It has a tree-like structure that starts with one main question called the root node which represents the entire dataset. From there, the tree branches out into different possibilities based on features in the data.

Root Node: Starting point representing the whole dataset.
Branches: Lines connecting nodes showing the flow from one decision to another.
Internal Nodes: Points where decisions are made based on data features.
Leaf Nodes: End points of the tree where the final decision or prediction is made.

A Decision Tree also helps with decision-making by showing possible outcomes clearly. 
By looking at the "branches" we can quickly compare options and figure out the best choice.

There are mainly two types of Decision Trees based on the target variable:

Classification Trees: Used for predicting categorical outcomes like spam or not spam. 
These trees split the data based on features to classify data into predefined categories.
Regression Trees: Used for predicting continuous outcomes like predicting house prices.
Instead of assigning categories, it provides numerical predictions based on the input features.
How Decision Trees Work?
1. Start with the Root Node: It begins with a main question at the root node which is derived from the dataset’s features.

2. Ask Yes/No Questions: From the root, the tree asks a series of yes/no questions to split the data into subsets based on specific attributes.

3. Branching Based on Answers: Each question leads to different branches:

If the answer is yes, the tree follows one path.
If the answer is no, the tree follows another path.
4. Continue Splitting: This branching continues through further decisions helps in reducing the data down step-by-step.

5. Reach the Leaf Node: The process ends when there are no more useful questions to ask leading to the leaf node where the final decision or prediction is made.

Let’s look at a simple example to understand how it works. Imagine we need to decide whether to drink coffee based on the time of day and how tired we feel. The tree first checks the time:

1. In the morning: It asks “Tired?”

If yes, the tree suggests drinking coffee.
If no, it says no coffee is needed.
2. In the afternoon: It asks again “Tired?”

If yes, it suggests drinking coffee.
If no, no coffee is needed.
'''

In [17]:
response = combined_chain.invoke(
    {'paragraph': text}
)

In [18]:
print(response)

Here is the merged document containing the paragraph notes and MCQs:

**Decision Trees: Mapping Choices and Outcomes**

Decision Trees are a powerful tool used in Machine Learning to map out different choices and their potential outcomes. They are particularly useful for tasks like classification (categorizing data) and prediction (forecasting future values).

The structure of a Decision Tree resembles a real tree, starting with a **Root Node** which represents the entire dataset. From this root, **Branches** extend, connecting different nodes and illustrating the flow of decisions. **Internal Nodes** act as decision points, where questions are asked based on specific features of the data. These questions lead to further branches, effectively splitting the dataset. The process continues until the **Leaf Nodes** are reached, which represent the final decision or prediction for a given data point.

Decision Trees are highly beneficial because they provide a clear visual representation of

In [19]:
combined_chain.get_graph().print_ascii()

              +--------------------------+             
              | Parallel<notes,mcq>Input |             
              +--------------------------+             
                   **              **                  
                ***                  ***               
              **                        **             
  +----------------+               +----------------+  
  | PromptTemplate |               | PromptTemplate |  
  +----------------+               +----------------+  
           *                                *          
           *                                *          
           *                                *          
+--------------------+               +------------+    
| GoogleGenerativeAI |               | ChatOpenAI |    
+--------------------+               +------------+    
           *                                *          
           *                                *          
           *                                *   

# Conditional Chain

In [20]:
from langchain.output_parsers import PydanticOutputParser

from pydantic import BaseModel, Field
from typing import Literal, Annotated

In [21]:
class Feedback(BaseModel):

    feedback: Annotated[str, Field(description = 'Feedback summary')]
    sentiment: Annotated[Literal['Positive', 'Negative'], Field(description = 'Classify the feedback into Positive or Negative')]

In [23]:
pydantic_parser = PydanticOutputParser(pydantic_object = Feedback)

In [26]:
temp1 = PromptTemplate(
    template = ''' Classify the below given feedback into Positive or Negative and also generate summary of the feedback.
    {feedback}
    {format_instructions}''',
    input_variables = ['feedback'],
    partial_variables = {'format_instructions': pydantic_parser.get_format_instructions()}
)


classifier_chain = temp1 | model_gpt | pydantic_parser

In [27]:
classifier_chain.invoke("This is the worst mobile.")

Feedback(feedback='This is the worst mobile.', sentiment='Negative')

In [32]:
result = classifier_chain.invoke(
    {'feedback': "This is the worst mobile phone ever"}
)

print(result)
print(result.feedback)
print(result.sentiment)

feedback='This is the worst mobile phone ever' sentiment='Negative'
This is the worst mobile phone ever
Negative


In [33]:
# Part 2: Now, we will create the condition's part

from langchain.schema.runnable import RunnableBranch, RunnableLambda

In [41]:
temp_pos = PromptTemplate(
    template = ''' Write an appropriate response to this positive feedback.
    {feedback}
    {sentiment}''',
    input_variables = ['feedback', 'sentiment']
)

temp_neg = PromptTemplate(
    template = ''' Write an appropriate response to this negative feedback.
    {feedback}
    {sentiment}''',
    input_variables = ['feedback', 'sentiment']
)

In [42]:
branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'Positive',
     RunnableLambda(lambda x: x.model_dump()) | temp_pos | model_gpt | parser),
    
    (lambda x: x.sentiment == 'Negative', 
     RunnableLambda(lambda x: x.model_dump()) | temp_neg | model_gpt | parser),
    
    RunnableLambda(lambda x: "Could not find the sentiment")
)

In [43]:
combined_chain = classifier_chain | branch_chain

response = combined_chain.invoke(
    {'feedback': 'This is the best mobile phone ever.'}

)

print(response)

Thank you so much for your kind words! We're thrilled to hear that you love your mobile phone. If you have any questions or need assistance, please don't hesitate to reach out. Enjoy your amazing device!


In [37]:
combined_chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOpenAI |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
